# 01 · Separación reproducible de los datos

Creamos una única separación **train / validación / test = 70 / 15 / 15**,
estratificada por `Churn`, con `random_state=42` en las dos llamadas a
`train_test_split`. Primero reservamos el 30 % y luego lo dividimos en mitades.
Por redondeo a filas enteras, validación y test pueden diferir en una fila.

Se conserva `customerID` para trazabilidad: **no será una feature**.
No se ajustan transformaciones ni modelos. El test solo se asigna y guarda;
no se calculan sus métricas ni se exploran sus variables.


In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd

RAIZ = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "CONTEXT.md").is_file() and (p / "src" / "limpieza.py").is_file()
)
import sklearn
from sklearn.model_selection import train_test_split

FUENTE = RAIZ / "archive" / "WA_Fn-UseC_-Telco-Customer-Churn_clean.csv"
DESTINO = RAIZ / "data" / "splits"
MANIFIESTO = DESTINO / "metadata.json"
SEMILLA = 42
huella_fuente = hashlib.sha256(FUENTE.read_bytes()).hexdigest()


## Crear una vez y reutilizar

Se comprueba que los clientes sean únicos, que no haya solapamientos y que
ninguna fila se pierda. Se guarda cada conjunto, la asignación por cliente,
las huellas SHA-256 y las versiones del entorno.

Si ya existe la separación, se reutiliza. Si cambia el CSV de origen, el proceso
se detiene para evitar mezclar experimentos con distintas versiones de datos.
No se sobreescriben automáticamente separaciones anteriores.


In [2]:
if MANIFIESTO.exists():
    metadata = json.loads(MANIFIESTO.read_text(encoding="utf-8"))
    if metadata["fuente_sha256"] != huella_fuente:
        raise ValueError("El CSV limpio cambió. Revisar la versión de la separación antes de continuar.")
    if metadata["random_state"] != SEMILLA or metadata["proporciones"] != [0.70, 0.15, 0.15]:
        raise ValueError("La configuración guardada no coincide con este experimento.")
    for nombre in metadata["archivos_sha256"]:
        if not (DESTINO / nombre).is_file():
            raise FileNotFoundError(f"Falta el archivo persistido: {nombre}")
    print("Se reutiliza la separación existente.")
else:
    datos = pd.read_csv(FUENTE, dtype=str)
    if datos["customerID"].isna().any() or not datos["customerID"].is_unique:
        raise ValueError("customerID debe ser único y no tener faltantes.")
    if datos["Churn"].isna().any() or set(datos["Churn"]) != {"No", "Yes"}:
        raise ValueError("Churn debe contener únicamente No/Yes, sin faltantes.")

    train, reserva = train_test_split(
        datos, test_size=0.30, stratify=datos["Churn"], random_state=SEMILLA
    )
    validacion, test_reservado = train_test_split(
        reserva, test_size=0.50, stratify=reserva["Churn"], random_state=SEMILLA
    )
    conjuntos = {"train": train, "validacion": validacion, "test": test_reservado}
    asignacion = pd.concat([
        parte[["customerID"]].assign(conjunto=nombre)
        for nombre, parte in conjuntos.items()
    ], ignore_index=True)
    assert len(asignacion) == len(datos)
    assert asignacion["customerID"].is_unique
    assert set(asignacion["customerID"]) == set(datos["customerID"])

    DESTINO.mkdir(parents=True, exist_ok=True)
    nombres = ["train.csv", "validacion.csv", "test.csv", "asignacion.csv"]
    if any((DESTINO / nombre).exists() for nombre in nombres):
        raise FileExistsError("Hay una separación incompleta; revisar antes de sobrescribirla.")
    huellas = {}
    for nombre, tabla in [
        *((f"{nombre}.csv", parte) for nombre, parte in conjuntos.items()),
        ("asignacion.csv", asignacion),
    ]:
        contenido = tabla.to_csv(index=False, lineterminator="\n").encode("utf-8")
        (DESTINO / nombre).write_bytes(contenido)
        huellas[nombre] = hashlib.sha256(contenido).hexdigest()
    metadata = {
        "fuente": FUENTE.relative_to(RAIZ).as_posix(),
        "fuente_sha256": huella_fuente,
        "random_state": SEMILLA,
        "proporciones": [0.70, 0.15, 0.15],
        "metodo": "train_test_split: reserva 30%; dividir reserva 50/50; estratificar ambas por Churn",
        "filas": {nombre: len(parte) for nombre, parte in conjuntos.items()},
        "versiones": {"pandas": pd.__version__, "scikit-learn": sklearn.__version__},
        "archivos_sha256": huellas,
    }
    MANIFIESTO.write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    del datos, train, reserva, validacion, test_reservado, conjuntos, asignacion, tabla
    print("Separación creada y guardada.")


Separación creada y guardada.


In [3]:
# Solo tamaños de asignación, sin estadísticas ni evaluación del test.
tamanos = pd.Series(metadata["filas"], name="clientes").to_frame()
tamanos["proporcion"] = tamanos["clientes"] / tamanos["clientes"].sum()
display(tamanos)


,clientes,proporcion
train,4930,0.699986
validacion,1056,0.149936
test,1057,0.150078


## Reutilización en futuros modelos

Cargar los CSV guardados, sin llamar otra vez a `train_test_split`.
Separar `Churn` como objetivo y excluir `customerID` de las features.
Ajustar imputadores, codificadores y escaladores **solo con train**.
Elegir modelos y umbrales con validación. El test queda reservado para la
evaluación final, cuando se autorice.
